In [1]:
# Cell 1: Load data and print a quick quality report
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.cleaning import report, normalise_categories, parse_dates, cap_outliers, find_outliers

df = pd.read_csv("../data/loan_book.csv")
print("==========BEFORE CLEANING==========")
report(df)

print()
df.info()

==========BEFORE CLEANING==========
Shape: 120960 rows, 26 columns

Missing values:
    annual_income: 8691 missing values
    employment_length_years: 3724 missing values
    num_open_accounts: 2437 missing values
    months_since_last_delinquency: 60380 missing values

Duplicate rows: 602

<class 'pandas.DataFrame'>
RangeIndex: 120960 entries, 0 to 120959
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   applicant_id_hash              120960 non-null  str    
 1   age                            120960 non-null  float64
 2   annual_income                  112269 non-null  float64
 3   employment_length_years        117236 non-null  float64
 4   home_ownership                 120960 non-null  str    
 5   region                         120960 non-null  str    
 6   num_open_accounts              118523 non-null  float64
 7   num_delinquencies_2yr          120960 non-null  int64

In [2]:
# Cell 2: Normalise categorical columns and parse dates
df = normalise_categories(df)
df = parse_dates(df)

print(df["home_ownership"].value_counts())
print(df["loan_purpose"].value_counts())
print(f"\n{df['application_date']}")


home_ownership
mortgage    54514
rent        42365
own         18150
other        5931
Name: count, dtype: int64
loan_purpose
debt_consolidation    33805
home_improvement      20553
major_purchase        18099
medical               14667
other                 14465
education             12189
small_business         7182
Name: count, dtype: int64

0        2021-06-14
1        2021-08-28
2        2021-02-23
3        2021-10-07
4        2021-08-23
            ...    
120955   2021-09-19
120956   2022-04-11
120957   2021-12-03
120958   2021-05-28
120959   2023-10-04
Name: application_date, Length: 120960, dtype: datetime64[us]


In [3]:
# Cell 3: Handle missing values
df["ever_delinquent"] = df["months_since_last_delinquency"].notna().astype(int)
df["months_since_last_delinquency"] = df["months_since_last_delinquency"].fillna(999)

df["num_open_accounts"] = df["num_open_accounts"].fillna(df["num_open_accounts"].median())
df["employment_length_years"] = df["employment_length_years"].fillna(df["employment_length_years"].median())

df["annual_income"] = df["annual_income"].fillna(df["annual_income"].median())

In [4]:
# Cell 4: Remove duplicate rows
print(f"Duplicate rows before: {df.duplicated().sum()}")
print(f"Rows before dropping duplicates: {len(df)}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after dropping duplicates: {len(df)}")

Duplicate rows before: 602
Rows before dropping duplicates: 120960
Rows after dropping duplicates: 120358


In [5]:
# Cell 5: Cap outliers on numeric columns using Winsorisation (IQR method)
# We cap rather than remove — preserves rows, just limits extreme values
numeric_cols = df.select_dtypes(include="number").columns.tolist()
exclude = ["default_flag", "branch_code_id", "months_at_current_address"]
cols_to_cap = [c for c in numeric_cols if c not in exclude]

for col in cols_to_cap:
    df = cap_outliers(df, col)

# Capping can create new duplicates — remove them
df = df.drop_duplicates().reset_index(drop=True)

print("==========AFTER CLEANING==========")
report(df)

==========AFTER CLEANING==========
Shape: 120344 rows, 27 columns

No missing values

Duplicate rows: 0


In [9]:
# Export cleaned dataset in csv format
df.to_csv("../outputs/loan_book_clean.csv", index=False)